# Data Cleaning

## Mounting Drive and Importing Libraries

In [1]:
# mounting drive
from google.colab import drive
drive.mount("/content/drive")

# importing libraries
import pandas as pd


Mounted at /content/drive


## Moth Data

### Primary Dataset: ECN_IM1.csv

In [2]:
# loading the dataset
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Data/Moth Data/ECN_IM1.csv"
df_moth1 = pd.read_csv(path)

# creating a copy for clean data
df_clean1 = df_moth1.copy()

# converting SDATE to datetime
df_clean1["SDATE"] = pd.to_datetime(df_clean1["SDATE"], errors = "coerce")

# removing entries with FIELDNAME "Q1"
df_clean1 = df_clean1[df_clean1["FIELDNAME"] != "Q1"]
df_clean1 = df_clean1.reset_index(drop = True)

# outliers to be handled later

# result
df_clean1

# outputting the data
df_clean1.to_csv("Primary Moth Data CLEAN.csv", index = False)


/tmp/ipython-input-3066369895.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean1["SDATE"] = pd.to_datetime(df_clean1["SDATE"], errors = "coerce")


### Secondary Data: ECN_IM2.csv

In [3]:
# loading the dataset
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Data/Moth Data/Supporting Documents/ECN_IM2.csv"
df_moth2 = pd.read_csv(path)

# creating a copy for clean data
df_clean2 = df_moth2.copy()

# converting FROMDATE and SDATE to datetime
df_clean2["FROMDATE"] = pd.to_datetime(df_clean2["FROMDATE"], errors = "coerce")
df_clean2["SDATE"] = pd.to_datetime(df_clean2["SDATE"], errors = "coerce")

# calculating WEEK values from SDATE - instead of imputing values for 0 and NaN in WEEK
df_clean2["WEEK_NEW"] = df_clean2["SDATE"].dt.strftime("%U").astype("int64") # calculating WEEK_NEW from SDATE and making it integer
df_clean2 = df_clean2.drop(columns = ["WEEK"]) # dropping original WEEK
df_clean2 = df_clean2[df_clean2["WEEK_NEW"] != 0].reset_index(drop = True)

# outliers to be handled later

# result
df_clean2

# outputting the data
df_clean2.to_csv("Secondary Moth Data CLEAN(traps).csv", index = False)


/tmp/ipython-input-1064001698.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean2["FROMDATE"] = pd.to_datetime(df_clean2["FROMDATE"], errors = "coerce")
/tmp/ipython-input-1064001698.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean2["SDATE"] = pd.to_datetime(df_clean2["SDATE"], errors = "coerce")


### Merging ECM_IM1.csv and ECM_IM2.csv

In [4]:
# checking for same columns (keys)
df_cols = list(df_clean1.columns)
df2_cols = list(df_clean2.columns)
same_cols = [] # empty list for same columns

for i in range(len(df_cols)):
  if df_cols[i] in df2_cols:
    same_cols.append(df_cols[i]) # appending same columns - identifiers

# merging
df_merged = df_clean1.merge(df_clean2, how = "left", on = same_cols)

# dropping rows that did not have matching data for weather variables
df_merged = df_merged.dropna()

# calculating outliers for every species (FIELDNAME) with all corresponding values up to 98th percentile
df_percentiles = df_merged.groupby(["FIELDNAME"])["VALUE"].quantile(0.98)

# adding a new column 98th percentile to the df_merged = adding percentile values for each FIELDNAME from df_percentiles as a new column
df_merged["98_PERCENTILE"] = df_merged["FIELDNAME"].map(df_percentiles)

# filtering out entries where FIELDNAME's VALUE is greater than 98th_PERCENTILE
df_merged = df_merged[df_merged["VALUE"] <= df_merged["98_PERCENTILE"]]

# resetting the index
df_merged = df_merged.reset_index(drop = True)

# changing SPERIOD_D, YEAR, WEEK_NEW to int64 datatype
df_merged[["SPERIOD_D", "YEAR", "WEEK_NEW"]] = df_merged[["SPERIOD_D", "YEAR", "WEEK_NEW"]].astype("int64")

# rounding 98_PERCENTILE to 2 decimal places
df_merged['98_PERCENTILE'] = round(df_merged["98_PERCENTILE"], 2)

# result
df_merged

# outputting the data
df_merged.to_csv("Moth Data Merged.csv", index = False)

## Meteorological Data

In [5]:
# loading the dataset
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Data/Meteorological Data/ECN_MA_1991_to_2015_merged.csv"
df_weather = pd.read_csv(path, index_col = 0)

# creating a copy for clean data
df_clean3 = df_weather.copy()

# removing year 1991 - moth data starts in year 1992
df_clean3["SDATE"] = pd.to_datetime(df_clean3["SDATE"], errors = "coerce") # converting SDATE to datetime
df_clean3 = df_clean3[df_clean3["SDATE"].dt.year != 1991]
df_clean3 = df_clean3.reset_index(drop = True)

# removing SITECODE T11 - SITECODE T11 not in moth data
df_clean3 = df_clean3[df_clean3["SITECODE"] != "T11"]
df_clean3 = df_clean3.reset_index(drop = True)

# SHOUR - modifying value 24 to 0
df_clean3.loc[(df_clean3["SHOUR"] == 24), "SHOUR"] = 0

# FIELDNAME - standardizing text
df_clean3["FIELDNAME"] = df_clean3["FIELDNAME"].astype(str).str.strip().str.upper()

# pivoting the table - moving FIELDNAME values to columns - reshaping the dataset
df_pivot = df_clean3.pivot_table(index = ["SITECODE", "AWSNO", "SDATE", "SHOUR"], columns = "FIELDNAME", values = "VALUE", aggfunc = "first")
df_pivot.columns.name = None # removing FIELDNAME from index column
df_pivot = df_pivot.reset_index()

# since there are 8 duplicate rows which capture same weather variable with same date date but different values, we only take first entry out of these duplicataes
# we will lose 8 rows of data where SITECODE, AWSNO, SDATE, SHOUR, and FIELDNAME are the same but the VALUE is not

# dropping columns SWATER_VWC, SWATER, DRYTMP_RH, RH, SWATER_T, ALBSKY, and WETTMP - huge amount of missing values after pivoting (over 20%)
df_pivot = df_pivot.drop(columns = ["SWATER_VWC", "SWATER", "DRYTMP_RH", "RH", "SWATER_T", "ALBSKY", "WETTMP"], axis = 1)

# collapsing df_pivot to one row per day - the variables across different hours are transferred to mean values measured over that day (because they are captured as average on each hour)
# RAIN has to be summed up
df_pivot = df_pivot.groupby(["SITECODE", "SDATE"]).agg({
    "ALBGRD": "mean",
    "DRYTMP": "mean",
    "NETRAD": "mean",
    "SOLAR": "mean",
    "STMP10": "mean",
    "STMP30": "mean",
    "SURWET": "mean",
    "WDIR": "mean",
    "WSPEED": "mean",
    "RAIN": "sum"
}).sort_values(by = "SDATE").reset_index()

# result
df_pivot

# outputting the pivoted weather data
df_pivot.to_csv("Pivoted Weather Data.csv", index = False)

/tmp/ipython-input-3038508114.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean3["SDATE"] = pd.to_datetime(df_clean3["SDATE"], errors = "coerce") # converting SDATE to datetime
